In [1]:
import torch
import triton
import triton.language as tl

In [2]:
def layernorm_forward_torch(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: torch.Tensor,
    eps: float = 1e-5,
):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    rstd = 1.0 / torch.sqrt(var + eps)
    x_hat = (x - mean) * rstd
    return x_hat * weight + bias

In [3]:
def _fwd_configs():
    return [
        triton.Config({"BLOCK_N": bn}, num_warps=nw)
        for bn in [256, 512, 1024, 2048, 4096]
        for nw in [2, 4, 8]
    ]

In [4]:
@triton.autotune(configs=_fwd_configs(), key=["N"])
@triton.jit
def _layernorm_fwd_kernel(
    X_ptr,
    W_ptr,
    B_ptr,
    Y_ptr,
    Mean_ptr,
    Rstd_ptr,
    M, N,
    stride_xm,
    stride_ym,
    eps: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    row = tl.program_id(0)

    X_row = X_ptr + row * stride_xm
    Y_row = Y_ptr + row * stride_ym

    mean_acc = 0.0
    for off in range(0, N, BLOCK_N):
        cols = off + tl.arange(0, BLOCK_N)
        mask = cols < N
        x = tl.load(X_row + cols, mask=mask, other=0.0).to(tl.float32)
        mean_acc += tl.sum(x, axis=0)
    mean = mean_acc / N

    var_acc = 0.0
    for off in range(0, N, BLOCK_N):
        cols = off + tl.arange(0, BLOCK_N)
        mask = cols < N
        x = tl.load(X_row + cols, mask=mask, other=0.0).to(tl.float32)
        diff = tl.where(mask, x - mean, 0.0)
        var_acc += tl.sum(diff * diff, axis=0)
    var = var_acc / N
    rstd = 1.0 / tl.sqrt(var + eps)

    tl.store(Mean_ptr + row, mean)
    tl.store(Rstd_ptr + row, rstd)

    for off in range(0, N, BLOCK_N):
        cols = off + tl.arange(0, BLOCK_N)
        mask = cols < N
        x = tl.load(X_row + cols, mask=mask, other=0.0).to(tl.float32)
        w = tl.load(W_ptr + cols, mask=mask, other=1.0).to(tl.float32)
        b = tl.load(B_ptr + cols, mask=mask, other=0.0).to(tl.float32)
        y = (x - mean) * rstd * w + b
        tl.store(Y_row + cols, y, mask=mask)

In [5]:
def _bwd_configs():
    return [
        triton.Config({"BLOCK_N": bn}, num_warps=nw)
        for bn in [256, 512, 1024, 2048, 4096]
        for nw in [2, 4, 8]
    ]


In [6]:
@triton.autotune(configs=_bwd_configs(), key=["N"])
@triton.jit
def _layernorm_bwd_kernel(
    DY_ptr,
    X_ptr,
    W_ptr,
    Mean_ptr,
    Rstd_ptr,
    DX_ptr,
    M, N,
    stride_m,
    BLOCK_N: tl.constexpr,
):
    row = tl.program_id(0)

    DY_row = DY_ptr + row * stride_m
    X_row = X_ptr + row * stride_m
    DX_row = DX_ptr + row * stride_m

    mean = tl.load(Mean_ptr + row).to(tl.float32)
    rstd = tl.load(Rstd_ptr + row).to(tl.float32)

    c1 = 0.0
    c2 = 0.0

    for off in range(0, N, BLOCK_N):
        cols = off + tl.arange(0, BLOCK_N)
        mask = cols < N
        dy = tl.load(DY_row + cols, mask=mask, other=0.0).to(tl.float32)
        x = tl.load(X_row + cols, mask=mask, other=0.0).to(tl.float32)
        w = tl.load(W_ptr + cols, mask=mask, other=1.0).to(tl.float32)
        x_hat = (x - mean) * rstd
        dxhat = dy * w
        c1 += tl.sum(tl.where(mask, dxhat, 0.0), axis=0)
        c2 += tl.sum(tl.where(mask, dxhat * x_hat, 0.0), axis=0)

    c1 = c1 / N
    c2 = c2 / N

    for off in range(0, N, BLOCK_N):
        cols = off + tl.arange(0, BLOCK_N)
        mask = cols < N
        dy = tl.load(DY_row + cols, mask=mask, other=0.0).to(tl.float32)
        x = tl.load(X_row + cols, mask=mask, other=0.0).to(tl.float32)
        w = tl.load(W_ptr + cols, mask=mask, other=1.0).to(tl.float32)
        x_hat = (x - mean) * rstd
        dxhat = dy * w
        dx = rstd * (dxhat - c1 - x_hat * c2)
        tl.store(DX_row + cols, dx, mask=mask)

In [7]:
@triton.jit
def _layernorm_dwdb_kernel(
    DY_ptr,
    X_ptr,
    Mean_ptr,
    Rstd_ptr,
    DW_ptr,
    DB_ptr,
    M,
    N,
    stride_dy_m,
    stride_dy_n,
    stride_x_m,
    stride_x_n,
    BLOCK_M: tl.constexpr,
):
    j = tl.program_id(0)
    acc_w = 0.0
    acc_b = 0.0
    rm = 0
    while rm < M:
        rows = rm + tl.arange(0, BLOCK_M)
        mask_r = rows < M
        dy = tl.load(
            DY_ptr + rows * stride_dy_m + j * stride_dy_n,
            mask=mask_r,
            other=0.0,
        ).to(tl.float32)
        x = tl.load(
            X_ptr + rows * stride_x_m + j * stride_x_n,
            mask=mask_r,
            other=0.0,
        ).to(tl.float32)
        mean = tl.load(Mean_ptr + rows, mask=mask_r, other=0.0).to(tl.float32)
        rstd = tl.load(Rstd_ptr + rows, mask=mask_r, other=0.0).to(tl.float32)
        x_hat = (x - mean) * rstd
        acc_w += tl.sum(tl.where(mask_r, dy * x_hat, 0.0), axis=0)
        acc_b += tl.sum(tl.where(mask_r, dy, 0.0), axis=0)
        rm += BLOCK_M
    tl.store(DW_ptr + j, acc_w)
    tl.store(DB_ptr + j, acc_b)

In [8]:
def layernorm_forward_triton(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: torch.Tensor,
    eps: float = 1e-5,
):
    assert x.is_cuda and weight.is_cuda and bias.is_cuda
    x = x.contiguous()
    shape = x.shape
    x2d = x.view(-1, shape[-1])
    M, N = x2d.shape

    y = torch.empty_like(x2d)
    mean = torch.empty(M, device=x.device, dtype=torch.float32)
    rstd = torch.empty(M, device=x.device, dtype=torch.float32)

    grid = (M,)
    _layernorm_fwd_kernel[grid](
        x2d, weight, bias, y, mean, rstd,
        M, N,
        x2d.stride(0), y.stride(0),
        eps,
    )
    return y.view(shape), mean, rstd

In [9]:
def layernorm_backward_triton(
    dy: torch.Tensor,
    x: torch.Tensor,
    weight: torch.Tensor,
    mean: torch.Tensor,
    rstd: torch.Tensor,
):
    dy = dy.contiguous()
    x = x.contiguous()
    shape = x.shape
    dy2d = dy.view(-1, shape[-1])
    x2d = x.view(-1, shape[-1])
    M, N = x2d.shape

    dx = torch.empty_like(x2d)
    dw = torch.empty(N, device=x.device, dtype=torch.float32)
    db = torch.empty(N, device=x.device, dtype=torch.float32)

    grid_m = (M,)
    _layernorm_bwd_kernel[grid_m](
        dy2d,
        x2d,
        weight,
        mean,
        rstd,
        dx,
        M,
        N,
        x2d.stride(0),
    )

    BLOCK_M = 128
    grid_n = (N,)
    _layernorm_dwdb_kernel[grid_n](
        dy2d,
        x2d,
        mean,
        rstd,
        dw,
        db,
        M,
        N,
        dy2d.stride(0),
        dy2d.stride(1),
        x2d.stride(0),
        x2d.stride(1),
        BLOCK_M,
    )
    return dx.view(shape), dw.to(weight.dtype), db.to(weight.dtype)

In [10]:
class LayerNormTriton(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, weight, bias, eps=1e-5):
        y, mean, rstd = layernorm_forward_triton(x, weight, bias, eps)
        ctx.save_for_backward(x, weight, mean, rstd)
        return y

    @staticmethod
    def backward(ctx, dy):
        x, weight, mean, rstd = ctx.saved_tensors
        dx, dw, db = layernorm_backward_triton(dy, x, weight, mean, rstd)
        return dx, dw, db, None

In [11]:
def triton_layer_norm(x, weight, bias, eps=1e-5):
    return LayerNormTriton.apply(x, weight, bias, eps)

In [12]:
def check_correctness():
    print("Correctness check")

    torch.manual_seed(42)
    M, N = 512, 1024
    dtype = torch.float32
    device = "cuda"

    x_ref = torch.randn(M, N, dtype=dtype, device=device, requires_grad=True)
    x_tri = x_ref.detach().clone().requires_grad_(True)
    weight = torch.randn(N, dtype=dtype, device=device, requires_grad=True)
    bias = torch.randn(N, dtype=dtype, device=device, requires_grad=True)
    w_tri = weight.detach().clone().requires_grad_(True)
    b_tri = bias.detach().clone().requires_grad_(True)

    y_ref = layernorm_forward_torch(x_ref, weight, bias)
    y_tri = triton_layer_norm(x_tri, w_tri, b_tri)

    torch.testing.assert_close(y_tri, y_ref, atol=1e-4, rtol=1e-4)
    print("Forward: OK")

    dy = torch.randn_like(y_ref)
    y_ref.backward(dy)
    y_tri.backward(dy)

    torch.testing.assert_close(x_tri.grad, x_ref.grad, atol=1e-4, rtol=1e-4)
    print("Backward: dx OK")
    torch.testing.assert_close(w_tri.grad, weight.grad, atol=1e-4, rtol=1e-4)
    print("Backward: dweight OK")
    torch.testing.assert_close(b_tri.grad, bias.grad, atol=1e-4, rtol=1e-4)
    print("Backward: dbias  OK")
    print()

In [13]:
def benchmark(M: int, N: int, n_warmup: int = 25, n_iter: int = 100):
    dtype = torch.float32
    device = "cuda"

    x = torch.randn(M, N, dtype=dtype, device=device, requires_grad=True)
    weight = torch.randn(N, dtype=dtype, device=device, requires_grad=True)
    bias = torch.randn(N, dtype=dtype, device=device, requires_grad=True)
    dy = torch.randn(M, N, dtype=dtype, device=device)

    ln = torch.nn.LayerNorm(N, device=device, dtype=dtype)
    ln.weight.data.copy_(weight)
    ln.bias.data.copy_(bias)

    def run_torch():
        y = ln(x)
        y.backward(dy)
        ln.zero_grad()
        if x.grad is not None:
            x.grad.zero_()

    def run_triton():
        xt = x.detach().requires_grad_(True)
        wt = weight.detach().requires_grad_(True)
        bt = bias.detach().requires_grad_(True)
        y = triton_layer_norm(xt, wt, bt)
        y.backward(dy)

    for _ in range(n_warmup):
        run_torch()
        run_triton()
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(n_iter):
        run_torch()
    end.record()
    torch.cuda.synchronize()
    t_torch = start.elapsed_time(end) / n_iter

    start.record()
    for _ in range(n_iter):
        run_triton()
    end.record()
    torch.cuda.synchronize()
    t_triton = start.elapsed_time(end) / n_iter

    speedup = t_torch / t_triton
    print(f"M={M:5d}, N={N:5d} | "
          f"PyTorch {t_torch:.3f} ms | "
          f"Triton  {t_triton:.3f} ms | "
          f"speedup {speedup:.2f}x")

In [14]:
def run_benchmarks():
    print("Benchmark (forward + backward, averaged over 100 iters)")
    configs = [
        (512, 768),
        (512, 1024),
        (512, 2048),
        (512, 4096),
        (2048, 768),
        (2048, 1024),
        (2048, 4096),
        (4096, 4096),
    ]
    for M, N in configs:
        benchmark(M, N)

In [17]:
check_correctness()

Correctness check
Forward: OK
Backward: dx OK
Backward: dweight OK
Backward: dbias  OK



In [16]:
run_benchmarks()

Benchmark (forward + backward, averaged over 100 iters)
M=  512, N=  768 | PyTorch 0.362 ms | Triton  0.636 ms | speedup 0.57x
M=  512, N= 1024 | PyTorch 0.346 ms | Triton  0.603 ms | speedup 0.57x
M=  512, N= 2048 | PyTorch 0.264 ms | Triton  0.566 ms | speedup 0.47x
M=  512, N= 4096 | PyTorch 0.489 ms | Triton  0.589 ms | speedup 0.83x
M= 2048, N=  768 | PyTorch 0.353 ms | Triton  0.584 ms | speedup 0.60x
M= 2048, N= 1024 | PyTorch 0.419 ms | Triton  0.673 ms | speedup 0.62x
M= 2048, N= 4096 | PyTorch 1.900 ms | Triton  2.340 ms | speedup 0.81x
M= 4096, N= 4096 | PyTorch 3.799 ms | Triton  4.708 ms | speedup 0.81x
